In [1]:
#Configuration
USE_EXCEL_INPUT = False  # Set to True to use an existing .xlsx file with AO3 links
EXCEL_FILE_PATH = "your_file.xlsx"  # Change to the path of the .xlsx file
COLUMN_NAME = "Link"  # Change to the name of the column containing the AO3 links

In [2]:
# File name: scraper.ipnyb
# This program allows you to scrape works from AO3
# Author: Maxim van der Maesen de Sombreff
# Date: 29-06-2025

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import csv
import time
import os
import pandas as pd
import re

"""
This script scrapes the full texts and selected metadata from Archive of Our Own (AO3).
This script allows for scraping from an existing list of URLs from an Excel file or a search query, set in the scipt itself.
The script allows for customization of the search query, but it is currently set to extract works from the Ancient Greek Religion and Lore fandom that contain rape/non-con.
The output of the script is the extracted texts as individual .txt files and an index as a .csv file.
"""

def get_session():
    """
    This function creates and configures a requests sesstion object with automatic retries and backoff.
    This function returns a session object configured to handle rate-limiting and connection drops.
    """

    session = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=2,
        status_forcelist=[500, 502, 503, 504, 525],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    return session

def clean_ao3_url(url):
    """
    This function extracts the base work ID form an AO3 URL by stripping out chapter elements from an URL if present.
    This function returns the extracted numeric work ID as a string or Non if no match is found.
    """

    match = re.search(r'works/(\d+)', str(url))
    if match:
        return match.group(1)
    return None

def save_work_details(session, work_id, headers, folder_name):
    """
    This function scrapes and saves the metadata and full text of a specific AO3 work.
    This function returns a dictionary containing the work's metadata or None if the request fails.
    This function also saves the text of the work as a .txt file in the specified folder.
    """

    url = f"https://archiveofourown.org/works/{work_id}?view_full_work=true"
    cookies = {'view_adult': 'true'}

    try:
        response = session.get(url, headers=headers, cookies=cookies, timeout=30)
        if response.status_code != 200:
            print(f"   [!] Failed to reach {work_id} (Status: {response.status_code})")
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        #Metadata extraction
        published_date = soup.find("dd", class_="published")
        published_date = published_date.text.strip() if published_date else "N/A"

        warning_section = soup.find("dd", class_="warning tags")
        warnings = "; ".join([t.text.strip() for t in warning_section.find_all("a", class_="tag")]) if warning_section else "None"

        words_elem = soup.find("dd", class_="words")
        words = words_elem.text.strip().replace(",", "") if words_elem else "0"

        chapters_elem = soup.find("dd", class_="chapters")
        chapters = chapters_elem.text.strip() if chapters_elem else "1/1"

        comments_elem = soup.find("dd", class_="comments")
        comments = comments_elem.text.strip() if comments_elem else "0"

        kudos_elem = soup.find("dd", class_="kudos")
        kudos = kudos_elem.text.strip() if kudos_elem else "0"

        hits_elem = soup.find("dd", class_="hits")
        hits = hits_elem.text.strip() if hits_elem else "0"

        #Content extraction
        content_parts = []
        main_container = soup.find("div", id="chapters")

        if main_container:
            chapters_list = main_container.find_all("div", class_="chapter")
            if chapters_list:
                for index, chapter in enumerate(chapters_list, start=1):
                    title_elem = chapter.find("h3", class_="title")
                    chapter_title = title_elem.get_text(strip=True) if title_elem else f"Chapter {index}"
                    text_container = chapter.find("div", class_="userstuff")
                    if text_container:
                        for landmark in text_container.find_all(class_="landmark"):
                            landmark.decompose()
                        chapter_body = text_container.get_text(separator="\n", strip=True)
                        content_parts.append(f"=== {chapter_title} ===\n\n{chapter_body}")
            else:
                text_container = main_container.find("div", class_="userstuff")
                if text_container:
                    for landmark in text_container.find_all(class_="landmark"):
                        landmark.decompose()
                    body = text_container.get_text(separator="\n", strip=True)
                    content_parts.append(f"=== Work Text ===\n\n{body}")

        if not content_parts:
            print(f"No content found for {work_id}.")
            return None

        #Standardized readable separator between chapters
        full_story_text = "\n\n" + ("\n\n" + "*" * 40 + "\n\n").join(content_parts)

        #Save as .txt File
        file_path = os.path.join(folder_name, f"{work_id}.txt")
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(f"WORK ID: {work_id}\nURL: {url}\nPUBLISHED: {published_date}\nWARNINGS: {warnings}\n")
            f.write(f"STATS -> Chapters: {chapters} | Words: {words} | Kudos: {kudos} | Comments: {comments} | Hits: {hits}\n")
            f.write("\n" + "=" * 60 + "\n\n")
            f.write(full_story_text)

        print(f"SAVED {work_id} (Ch: {chapters}, Words: {words})")

        return {
            "Work ID": work_id,
            "URL": url,
            "Published": published_date,
            "Warnings": warnings,
            "Chapters": chapters,
            "Words": words,
            "Kudos": kudos,
            "Comments": comments,
            "Hits": hits,
            "File Name": f"{work_id}.txt"
        }

    except Exception as e:
        print(f"Error processing {work_id}: {e}")
        return None

def run_scraper(total_pages):
    """
    This function executes the main scraping routine.
    It creates the output directory, initializes the web session, compiles a list of work IDs,
    and iteratively downloads the works.
    This function saves an index/summary of scraped metadata to '00_index.csv'.
    """

    folder_name = "output_folder_name"
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    session = get_session()
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}

    all_work_ids = []

    #Use links from excel file if USE_EXCEL_INPUT is set to TRUE
    if USE_EXCEL_INPUT:
        print(f"LOADING LINKS FROM {EXCEL_FILE_PATH}")
        df = pd.read_excel(EXCEL_FILE_PATH)
        raw_links = df[COLUMN_NAME].dropna().unique()
        for link in raw_links:
            wid = clean_ao3_url(link)
            if wid: all_work_ids.append(wid)
        print(f"Found {len(all_work_ids)} work IDs to scrape.")
    else:
        search_url = "https://archiveofourown.org/works/search"
        params = {
            "work_search[fandom_names]": "Ancient Greek Religion & Lore",
            "work_search[archive_warning_ids][]": "19",
            "work_search[language_id]": "en",
            "work_search[sort_column]": "_score",
            "commit": "Search"
        }
        for page in range(1, total_pages + 1):
            print(f"\nSEARCH PAGE {page}")
            params['page'] = page
            res = session.get(search_url, params=params, headers=headers, timeout=30)
            soup = BeautifulSoup(res.text, "html.parser")
            work_lis = soup.find_all("li", id=lambda x: x and x.startswith('work_'))
            all_work_ids.extend([li.get('id').replace("work_", "") for li in work_lis])
            if not work_lis: break

    processed_data = []
    for work_id in all_work_ids:
        if os.path.exists(os.path.join(folder_name, f"{work_id}.txt")):
            print(f"SKIPPING {work_id} (Already exists in the output folder)")
            continue

        data = save_work_details(session, work_id, headers, folder_name)
        if data:
            processed_data.append(data)

            csv_path = os.path.join(folder_name, "00_index.csv")
            file_exists = os.path.isfile(csv_path)
            with open(csv_path, 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=["Work ID", "URL", "Published", "Warnings", "Chapters", "Words", "Kudos", "Comments", "Hits", "File Name"])
                if not file_exists: writer.writeheader()
                writer.writerow(data)

        time.sleep(10)

In [ ]:
#Run the scraper
total_pages = 1  # Change this to the desired number of pages to scrape
run_scraper(total_pages)

In [ ]:
!zip -r /content/data.zip /content/output_folder_name